# 10 - Audit response: the kappa control-task null and the missing variance-fraction sweeps
## 1. The closure fraction's control-task null
## 2. The variance-fraction sweep on the control and position arms

## Before you run

Enable internet access for the dataset downloads. Attach the files below under a single `results/` directory.

| Needed by | What | Size |
|---|---|---|
| 5 | `results/probes_pinned/{color_strong,control_strong}/{stacks.npz,meta.json}` | 164 KB |
| 6 | `results/encoders/control/control_strong_seed*/backbone*.pt` (12) | ~0.5 GB |
| 6 | `results/encoders/position/position_strong_seed*/backbone*.pt` (12) | ~0.5 GB |
| 6 | `results/displacement/threshold_calibration.json` | 4 KB |
| 6b | `results/encoders/standard_strong_seed1/standard_strong_seed1.pt` | 44 MB |
| 7 | `results/displacement/*.json`, `results/hypotheses/kappa_ci.json` | 1.6 MB |
| 7 | `results/probes_pinned/`, `results/probes/`, `results/probes_excluded/` | 364 KB |

Section 5 builds the untrained encoders from seeds.

**Runtime**, on a T4. Roughly half on a P100 or better.

| Section | Work | Estimate |
|---|---|---|
| 3 | Shapes3D 255 MB download, decompress to a 5.6 GB memmap; dSprites 26 MB | 15-25 min |
| 5 | 504k embeddings + 2,304 probe fits (12 seeds x 6 factors x 4 rungs x 4 weight decays x 2 label sets) | 25-45 min |
| 6 | 8.0M embeddings (3.3M Shapes3D control, 4.7M dSprites position, over 4 runs) | 1.5-2.5 h |
| 6b | crop+colour encoder: 42k embeddings + 96 probe fits, then 69k for its displacement | 5-15 min |
| 7 | re-analysis and figures, CPU | < 2 min |

**Total 2.5-3.5 h**, inside one 9-hour session and about a tenth of the weekly GPU quota.

**Run them as two sessions if you prefer.** They are independent. Section 5 is the one that could
change a claim in the paper and it needs no checkpoints, so it is the cheaper thing to settle first:
attach only the two `probes_pinned` cells, run sections 0-5, and stop. **Section 6b is the other
cheap one**: it needs a single 44 MB checkpoint and about ten minutes, and it speaks to the paper's
largest structural weakness, so it is worth pairing with section 5. Section 6 is the expensive one
and only tightens a sensitivity already reported as unswept.

**Before Save Version**, delete the image caches or the output snapshot carries 5.6 GB of
regenerable data:

```
rm -f data/raw/3dshapes.images.npy data/raw/dsprites.images.npy
```

## 0. GPU

In [ ]:
!nvidia-smi

## 1. Repo

Set `PROBE_REPO_URL` to your public Git URL or `PROBE_REPO_DIR` to a local checkout.

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_DIR = os.environ.get("PROBE_REPO_DIR", "/kaggle/working/probe-capacity-invariance")
if not (Path(REPO_DIR) / "pyproject.toml").is_file():
    repo_url = os.environ.get("PROBE_REPO_URL")
    if not repo_url:
        raise RuntimeError("Set PROBE_REPO_URL or PROBE_REPO_DIR before running this notebook.")
    subprocess.run(["git", "clone", repo_url, REPO_DIR], check=True)
os.chdir(REPO_DIR)


## 2. Dependencies

In [ ]:
!pip install -q -e . --no-deps
!pip install -q h5py

import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
assert torch.cuda.is_available(), "no GPU - switch the accelerator on before running"

## 3. Datasets

In [ ]:
!python -m src.data.shapes3d --download --build-cache
!python -m src.data.dsprites --download --build-cache

## 4. Attach the prior results

In [ ]:









import hashlib, json, shutil
from pathlib import Path

REPO, INPUT, SEP = Path("/kaggle/working/probe-capacity-invariance"), Path("/kaggle/input"), "__"

def sha256(p, blocksize=1 << 20):
    h = hashlib.sha256()
    with open(p, "rb") as fh:
        for b in iter(lambda: fh.read(blocksize), b""):
            h.update(b)
    return h.hexdigest()

manifests = list(INPUT.rglob("MANIFEST.json"))
expected = {}
if manifests:
    man = json.loads(manifests[0].read_text())
    expected = {m["upload_name"]: m for m in man["files"]}
    print(f"manifest: {manifests[0].relative_to(INPUT)}, {len(expected)} files expected")
else:
    print("no MANIFEST.json found — falling back to name decoding without verification")

placed, verified, corrupt, skipped = 0, 0, [], 0
for src in INPUT.rglob("*"):
    if not src.is_file() or src.name == "MANIFEST.json" or src.name == "README.txt":
        continue
    if SEP in src.name:
        rel = src.name.replace(SEP, "/")
    elif "results" in src.parts:
        rel = str(Path(*src.parts[src.parts.index("results"):]))
    else:
        skipped += 1
        continue
    meta = expected.get(src.name)
    if meta is not None:
        if src.stat().st_size != meta["bytes"] or sha256(src) != meta["sha256"]:
            corrupt.append(src.name)
            continue
        rel, ok = meta["destination"], True
        verified += 1
    dst = REPO / rel
    dst.parent.mkdir(parents=True, exist_ok=True)
    if not dst.exists() or dst.stat().st_size != src.stat().st_size:
        shutil.copy2(src, dst)
    placed += 1

print(f"placed {placed} file(s); {verified} verified against the manifest; "
      f"{skipped} unrecognized file(s) ignored")
for c in corrupt:
    print(f"  CORRUPT (not placed): {c}")
if expected:
    absent = [n for n in expected if not (REPO / expected[n]["destination"]).exists()]
    for n in absent:
        print(f"  NOT UPLOADED: {n}  -> {expected[n]['destination']} "
              f"(section {expected[n]['section']})")
    print(f"{len(expected) - len(absent)}/{len(expected)} manifest files in place")
assert not corrupt, "re-upload the corrupt files before running anything below"

In [ ]:

from pathlib import Path

REPO = Path("/kaggle/working/probe-capacity-invariance")
need = {
    "5": ["results/probes_pinned/color_strong/stacks.npz",
          "results/probes_pinned/color_strong/meta.json",
          "results/probes_pinned/control_strong/stacks.npz",
          "results/probes_pinned/control_strong/meta.json"],
    "6": ["results/displacement/threshold_calibration.json"],
    "7": ["results/hypotheses/kappa_ci.json",
          "results/displacement/color_strong.json",
          "results/displacement/control_strong.json",
          "results/displacement/position_strong.json",
          "results/probes_pinned/color_weak/stacks.npz",
          "results/probes_pinned/position_strong/stacks.npz",
          "results/probes/color_strong/meta.json",
          "results/probes_excluded/scale_strong/meta.json",
          "results/probes_excluded/orientation_strong/meta.json",
          "results/probes_excluded/scale_recipe_strong/meta.json"],
}
missing = {s: [f for f in fs if not (REPO / f).exists()] for s, fs in need.items()}
for s, fs in need.items():
    got = len(fs) - len(missing[s])
    print(f"section {s}: {got}/{len(fs)} present")
    for f in missing[s]:
        print(f"    MISSING  {f}")

ckpt = {a: sorted((REPO / "results/encoders" / a).glob(f"{a}_strong_seed*/backbone*.pt"))
        for a in ("control", "position")}
for a, ps in ckpt.items():
    print(f"section 6: {a} backbones {len(ps)}/12")
    seeds = sorted(int(p.parent.name.split("seed")[-1]) for p in ps)
    if len(ps) != len(set(seeds)):
        print("    WARNING: more than one backbone in some seed directory; the glob would "
              "score a seed twice. Keep one file per directory.")
    if ps and seeds != list(range(12)):
        print(f"    seeds present: {seeds}")

std = REPO / "results/encoders/standard_strong_seed1/standard_strong_seed1.pt"
print(f"section 6b: {'OK  ' if std.exists() else 'MISSING'} {std.name}")

print()
if not missing["5"]:
    print("READY: section 5 can run.")
else:
    print("BLOCKED: section 5 is missing inputs listed above.")
if not missing["6"] and all(len(v) == 12 for v in ckpt.values()):
    print("READY: section 6 can run.")
else:
    print("BLOCKED: section 6 needs the 24 backbones and the calibration file.")
print("READY: section 6b can run." if std.exists() and not missing["6"]
      else "BLOCKED: section 6b needs standard_strong_seed1.pt and the calibration file.")

## 5. The kappa control-task null

In [ ]:
!python -m src.eval.kappa_null \
    --dataset shapes3d --cells color_strong control_strong \
    --device cuda --batch-size 512 --num-workers 2 \
    --out results/hypotheses/kappa_null.json

In [ ]:
import json
from pathlib import Path

r = json.loads(Path("results/hypotheses/kappa_null.json").read_text())
print("device check:", {k: (round(v, 5) if isinstance(v, float) else v)
                        for k, v in r["device_check"].items() if k != "note"})
print()
hdr = f"{'cell':16s} {'factor':12s} {'kappa':>22s} {'kappa_perm (null)':>24s}"
print(hdr); print("-" * len(hdr))
for cell, rows in r["cells"].items():
    for w in rows:
        k = (f"{w['kappa']:+.3f} [{w['kappa_ci95'][0]:+.3f},{w['kappa_ci95'][1]:+.3f}]"
             if w["kappa_defined"] else "undefined")
        kp = (f"{w['kappa_perm']:+.3f} [{w['kappa_perm_ci95'][0]:+.3f},"
              f"{w['kappa_perm_ci95'][1]:+.3f}]" if w["kappa_perm_defined"] else "undefined")
        print(f"{cell:16s} {w['factor']:12s} {k:>22s} {kp:>24s}")

### Verdict on the null

In [ ]:

hues = ("floor_hue", "wall_hue", "object_hue")
get = lambda c, f: next(w for w in r["cells"][c] if w["factor"] == f)
defined = all(get(c, f)["kappa_perm_defined"] for c in ("control_strong", "color_strong")
              for f in hues)
split_real = all(get("control_strong", f)["kappa"] > 0.5 and get("color_strong", f)["kappa"] < 0.1
                 for f in hues)
split_null = defined and all(
    get("control_strong", f)["kappa_perm"] > 0.5 and get("color_strong", f)["kappa_perm"] < 0.1
    for f in hues)
print("reported dissociation present :", split_real)
print("null reproduces it            :", split_null if defined else "kappa_perm undefined")
print()
print("BRANCH:", "RESCOPE Section 5.4" if split_null else
      ("Section 5.4 STANDS" if defined else "inspect kappa_perm by hand - undefined rows"))

## 6. The missing variance-fraction sweeps

In [ ]:
for vf in (0.95, 0.999):
    !python -m src.probes.displacement \
        --dataset shapes3d --cell control_strong_vf{vf} --amendment A11-2026-08-13 \
        --trained-encoders results/encoders/control/control_strong_seed*/backbone*.pt \
        --random-seed 0 1 2 3 4 5 6 7 8 9 10 11 --var-fraction {vf} \
        --n-contexts 512 --no-pixel-reference --device cuda --num-workers 2 \
        --out-root results/displacement

In [ ]:
for vf in (0.95, 0.999):
    !python -m src.probes.displacement \
        --dataset dsprites --cell position_strong_vf{vf} --amendment A11-2026-08-13 \
        --trained-encoders results/encoders/position/position_strong_seed*/backbone*.pt \
        --random-seed 0 1 2 3 4 5 6 7 8 9 10 11 --var-fraction {vf} \
        --n-contexts 512 --no-pixel-reference --device cuda --num-workers 2 \
        --out-root results/displacement

## 6b. The crop-plus-colour encoder that already exists

In [ ]:
!python -m src.probes.run_sweep --config configs/probe/ladder.yaml \
    --dataset shapes3d --condition standard --strength strong \
    --encoders results/encoders/standard_strong_seed1/standard_strong_seed1.pt \
    --random-seed 0 1 2 3 4 5 6 7 8 9 10 11 \
    --subsample 2000 --probe-steps 1000 --regime interpolation \
    --device cuda --num-workers 2 --out-root results/probes_exploratory

In [ ]:

import json
import numpy as np
from pathlib import Path

def cell(root, name):
    z = np.load(Path(root) / name / "stacks.npz")
    m = json.loads((Path(root) / name / "meta.json").read_text())
    return z, [f["name"] for f in m["factors"]]

hues = ("floor_hue", "wall_hue", "object_hue")
rows = {}
for root, name, label in (("results/probes_pinned", "control_strong", "control (crop only)"),
                          ("results/probes_pinned", "color_strong", "colour (photometric only)"),
                          ("results/probes_exploratory", "standard_strong", "standard (crop+colour)")):
    try:
        z, f = cell(root, name)
    except FileNotFoundError:
        print(f"missing: {root}/{name}"); continue
    rows[label] = {h: dict(R=z["trained"].mean(0)[f.index(h)],
                           Rp=z["perm"].mean(0)[f.index(h)],
                           Rr=z["random"].mean(0)[f.index(h)]) for h in hues}

for h in hues:
    print(f"\n=== {h} ===")
    print(f"  {'arm':28s} {'R linear':>9s} {'R deep':>9s} {'R_perm deep':>12s} {'kappa':>8s}")
    for label, d in rows.items():
        r, rp, rr = d[h]["R"], d[h]["Rp"], d[h]["Rr"]
        D0, D3 = rr[0] - r[0], rr[3] - r[3]
        k = 1 - D3 / D0 if D0 > 0 else float("nan")
        print(f"  {label:28s} {r[0]:9.3f} {r[3]:9.3f} {rp[3]:12.3f} {k:8.3f}")

In [ ]:

if "standard (crop+colour)" in rows:
    s = rows["standard (crop+colour)"]
    neg_R = all(s[h]["R"][3] < 0 for h in hues)
    dead_control = all(s[h]["Rp"][3] < 0 for h in hues)
    print("hue R^2 negative at the deepest rung      :", neg_R)
    print("control task unrecoverable at every rung  :", dead_control)
    print()
    if neg_R and dead_control:
        print("BRANCH: the crop does NOT rescue hue. Removing the crop is not what destroys it,")
        print("        so the S1 confound is not a live alternative. Say so in Section 5.4 and")
        print("        in the Limitations entry, citing this as exploratory n=1.")
    elif not neg_R and not dead_control:
        print("BRANCH: hue survives once a crop is present. The confound is REAL and the")
        print("        dissociation must be restated as a crop effect, not a photometric one.")
    else:
        print("BRANCH: mixed. Report both signatures and claim nothing about the confound.")

### Optional: score its displacement too

In [ ]:
!python -m src.probes.displacement \
    --dataset shapes3d --cell standard_strong --amendment A11-2026-08-13 \
    --trained-encoders results/encoders/standard_strong_seed1/standard_strong_seed1.pt \
    --random-seed 0 1 2 3 4 5 6 7 8 9 10 11 --var-fraction 0.99 \
    --n-contexts 512 --no-pixel-reference --device cuda --num-workers 2 \
    --out-root results/displacement

## 7. Rebuild the rank-stability companion and the figure

In [ ]:
!python -m src.eval.displacement_scale

In [ ]:
!python -m src.eval.figures --root results/probes_pinned --out results/figures/real \
    --claims-only --audit-figures \
    --kappa-json results/hypotheses/kappa_ci.json \
    --displacement-dir results/displacement

## 8. What to bring home